### Save-Load Example
Each Cell runs independently of the state of the Notebook. (With the expection of cell one which has to be run first for common variables and imports)
This shows the workflow more closely resembling actual use in a real project.

In [1]:
from federated_rsf.models import LocalRandomSurvivalForest, FederatedRandomSurvivalForest
from sksurv.datasets import load_aids, load_breast_cancer, load_flchain, load_gbsg2, load_whas500, load_veterans_lung_cancer
from federated_rsf.schema import DatasetSchema, SchemaAligner, SchemaCreator
from federated_rsf.testing import federate_data
from sksurv.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import pickle
import os

dataset_names = ['aids', 'breast_cancer', 'flchain', 'gbsg2', 'whas500', 'veterans_lung_cancer']
dataset = dataset_names[0]

client_count = 5

update_methods=['all', 'constant']
update_method = update_methods[1]

test_size = 0.3

save_folder = 'save_load'

In [2]:
dataset_name_to_loader = {
    'aids': load_aids,
    'breast_cancer': load_breast_cancer,
    'flchain': load_flchain,
    'gbsg2': load_gbsg2,
    'whas500': load_whas500,
    'veterans_lung_cancer': load_veterans_lung_cancer
}

X, Y = dataset_name_to_loader[dataset]()

# save X and Y to pickle files
os.makedirs(save_folder, exist_ok=True)
with open(os.path.join(save_folder, 'X.pkl'), 'wb') as f:
    pickle.dump(X, f)
with open(os.path.join(save_folder, 'Y.pkl'), 'wb') as f:
    pickle.dump(Y, f)

def convert_to_federated_datasets(X, Y, client_count, random_state=0, shuffle=True):
    X_list, Y_list = federate_data(X, Y, clients=client_count, random_state=random_state, shuffle=shuffle)
    X_list = [OneHotEncoder().fit_transform(X) for X in X_list]

    schema_list = [DatasetSchema(X.columns) for X in X_list]
    schema_creator = SchemaCreator()
    federated_schemas = schema_creator.fit_transform(schema_list)

    X_aligned_list = []
    local_schema_aligners = []
    for X_local, schema in zip(X_list, federated_schemas):
        aligner = SchemaAligner().fit(schema)
        X_aligned = aligner.transform(X_local)
        X_aligned_list.append(X_aligned)
        local_schema_aligners.append(aligner)
    return X_aligned_list, Y_list

X_list, Y_list = convert_to_federated_datasets(X, Y, client_count, random_state=0)

os.makedirs(save_folder, exist_ok=True)
with open(os.path.join(save_folder, 'X_list.pkl'), 'wb') as f:
    pickle.dump(X_list, f)
with open(os.path.join(save_folder, 'Y_list.pkl'), 'wb') as f:
    pickle.dump(Y_list, f)

In [3]:
local_models = [LocalRandomSurvivalForest(random_state=0, update_method=update_method) for _ in range(client_count)]
global_model = LocalRandomSurvivalForest(random_state=0, update_method=update_method)

def split_data(X_list, Y_list, test_size):
    X_trains, X_tests, Y_trains, Y_tests = [], [], [], []

    for X_local, Y_local in zip(X_list, Y_list):

        X_train, X_test, Y_train, Y_test = train_test_split(X_local, Y_local, test_size=test_size, random_state=0)
        X_trains.append(X_train)
        X_tests.append(X_test)
        Y_trains.append(Y_train)
        Y_tests.append(Y_test)

    return X_trains, Y_trains, X_tests, Y_tests

def fit_and_federate_models(X_trains, Y_trains, local_models):
    for X_train, Y_train, local_model in zip(X_trains, Y_trains, local_models):
        local_model.fit(X_train, Y_train)

    federated_model = FederatedRandomSurvivalForest(local_models=local_models)
    federated_model.distribute_trees()

    return local_models

def train_global_model(X_trains, Y_trains, global_model):
    X_global_train = pd.concat(X_trains, ignore_index=True)
    Y_global_train = np.concatenate(Y_trains)

    global_model.fit(X_global_train, Y_global_train)

    return global_model

# load x and y lists from pickle files
with open(os.path.join(save_folder, 'X_list.pkl'), 'rb') as f:
    X_list = pickle.load(f)
with open(os.path.join(save_folder, 'Y_list.pkl'), 'rb') as f:
    Y_list = pickle.load(f)

X_trains, Y_trains, X_tests, Y_tests = split_data(X_list, Y_list, test_size)

# save train and test datasets to pickle files
with open(os.path.join(save_folder, 'X_trains.pkl'), 'wb') as f:
    pickle.dump(X_trains, f)
with open(os.path.join(save_folder, 'Y_trains.pkl'), 'wb') as f:
    pickle.dump(Y_trains, f)
with open(os.path.join(save_folder, 'X_tests.pkl'), 'wb') as f:
    pickle.dump(X_tests, f)
with open(os.path.join(save_folder, 'Y_tests.pkl'), 'wb') as f:
    pickle.dump(Y_tests, f)

local_models = fit_and_federate_models(X_trains, Y_trains, local_models)
global_model = train_global_model(X_trains, Y_trains, global_model)

# save local and global models to pickle files
with open(os.path.join(save_folder, 'local_models.pkl'), 'wb') as f:
    pickle.dump(local_models, f)
with open(os.path.join(save_folder, 'global_model.pkl'), 'wb') as f:
    pickle.dump(global_model, f)

In [4]:
def evaluate_models_text(local_models, global_model, X_tests, Y_tests):
    for i, (local_model, X_test, Y_test) in enumerate(zip(local_models, X_tests, Y_tests)):
        global_c_index = global_model.score(X_test, Y_test)
        print(f"Client {i+1}")
        print(f"Global model C-index: {global_c_index:.4f}")
        local_model.use_local_estimators()
        c_index = local_model.score(X_test, Y_test)
        print(f"Local model C-index: {c_index:.4f}")
        local_model.use_federated_estimators()
        federated_c_index = local_model.score(X_test, Y_test)
        if global_c_index < federated_c_index > c_index:
            color = "\033[32m"  # Green
        elif global_c_index >= federated_c_index <= c_index:
            color = "\033[31m"  # Red
        else:
            color = "\033[33m"  # Yellow
        print(f"Federated model C-index: {color}{federated_c_index:.4f}\033[0m")
        print(f"Number of estimators in federated model: {local_model.n_estimators}")
        print("-" * 30)

#load local and global models from pickle files
with open(os.path.join(save_folder, 'local_models.pkl'), 'rb') as f:
    local_models = pickle.load(f)
with open(os.path.join(save_folder, 'global_model.pkl'), 'rb') as f:
    global_model = pickle.load(f)

#load test datasets from pickle files
with open(os.path.join(save_folder, 'X_tests.pkl'), 'rb') as f:
    X_tests = pickle.load(f)
with open(os.path.join(save_folder, 'Y_tests.pkl'), 'rb') as f:
    Y_tests = pickle.load(f)

evaluate_models_text(local_models, global_model, X_tests, Y_tests)

Client 1
Global model C-index: 0.4859
Local model C-index: 0.5803
Federated model C-index: 0.7155
Number of estimators in federated model: 100
------------------------------
Client 2
Global model C-index: 0.7483
Local model C-index: 0.7552
Federated model C-index: 0.7483
Number of estimators in federated model: 100
------------------------------
Client 3
Global model C-index: 0.8737
Local model C-index: 0.8535
Federated model C-index: 0.8384
Number of estimators in federated model: 100
------------------------------
Client 4
Global model C-index: 0.6211
Local model C-index: 0.6499
Federated model C-index: 0.6643
Number of estimators in federated model: 100
------------------------------
Client 5
Global model C-index: 0.6854
Local model C-index: 0.7219
Federated model C-index: 0.7285
Number of estimators in federated model: 100
------------------------------
